# Hospital Readmission Risk Prediction
## 03. Experiment Tracking — Scenario 1: Baseline (Logistic Regression)

Per proposal 4.2–4.3: Logistic Regression serves as the **baseline**. A model is promoted only if it exceeds the baseline on validation **PR-AUC**.

### MLflow setup (local)
- **Tracking**: SQLite (`mlflow.db`)
- **Metrics**: PR-AUC (primary), ROC-AUC, Recall at K
- **Artifacts**: Model, confusion matrix heatmap

In [6]:
%pip install mlflow pandas scikit-learn xgboost lightgbm seaborn matplotlib

Note: you may need to restart the kernel to use updated packages.


In [7]:
import mlflow
mlflow.set_tracking_uri("sqlite:///mlflow.db")
print(f"Tracking URI: {mlflow.get_tracking_uri()}")

Tracking URI: sqlite:///mlflow.db


## Load Data & Prepare Features
Reuse logic from 02: patient-level split, feature engineering.

In [8]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score, confusion_matrix
from sklearn.model_selection import train_test_split

DATA_PATH = '../data/diabetic_data.csv'
TARGET = 'target'
SEED = 42

def read_data(path, limit=None):
    df = pd.read_csv(path)
    df['target'] = df['readmitted'].isin(['30', '<30']).astype(int)
    if 'weight' in df.columns:
        df = df.drop(columns=['weight'])
    for col in ['medical_specialty', 'payer_code']:
        if col in df.columns:
            df[col] = df[col].fillna('Unknown').replace('?', 'Unknown')
    df['age'] = df['age'].fillna('[50-60)')
    df['gender'] = df['gender'].fillna('Unknown')
    df['change'] = df['change'].fillna('No')
    df['diabetesMed'] = df['diabetesMed'].fillna('No')
    for col in ['number_emergency', 'number_inpatient', 'number_outpatient']:
        df[col] = pd.to_numeric(df.get(col, 0), errors='coerce').fillna(0).astype(int)
    df['care_intensity'] = df['number_emergency'] + df['number_inpatient'] + df['number_outpatient']
    df['medication_changed'] = (df['change'] == 'Ch').astype(int)
    for col in ['num_lab_procedures', 'num_procedures', 'num_medications', 'number_diagnoses']:
        df[col] = pd.to_numeric(df.get(col, 0), errors='coerce').fillna(0).astype(int)
    for col in ['A1Cresult', 'max_glu_serum']:
        if col not in df.columns:
            df[col] = 'not_tested'
        else:
            df[col] = df[col].fillna('None').replace('None', 'not_tested').astype(str)
    if 'race' in df.columns:
        df['race'] = df['race'].fillna('Unknown').replace('?', 'Unknown').astype(str)
    if limit:
        df = df.sample(n=min(limit, len(df)), random_state=SEED)
    return df

FEATURE_COLS = ['time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications',
    'number_emergency', 'number_inpatient', 'number_outpatient', 'number_diagnoses', 'care_intensity',
    'admission_type_id', 'discharge_disposition_id', 'admission_source_id',
    'age', 'gender', 'race', 'change', 'diabetesMed', 'medication_changed', 'A1Cresult', 'max_glu_serum']

df = read_data(DATA_PATH, limit=50_000)
FEATURE_COLS = [c for c in FEATURE_COLS if c in df.columns]
patient_target = df.groupby('patient_nbr')[TARGET].max()
train_patients, val_patients = train_test_split(
    patient_target.index.tolist(), test_size=0.2, random_state=SEED, stratify=patient_target.values
)
df_train = df[df['patient_nbr'].isin(train_patients)]
df_val = df[df['patient_nbr'].isin(val_patients)]

train_dicts = df_train[FEATURE_COLS].to_dict(orient='records')
val_dicts = df_val[FEATURE_COLS].to_dict(orient='records')
dv = DictVectorizer()
X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)
y_train = df_train[TARGET].values
y_val = df_val[TARGET].values

print(f'Train: {X_train.shape[0]:,} rows; Val: {X_val.shape[0]:,} rows')
print(f'Positive rate (train): {y_train.mean():.2%}')

Train: 40,030 rows; Val: 9,970 rows
Positive rate (train): 11.15%


## Train Baseline & Log to MLflow
Logistic Regression with `class_weight='balanced'`. Log PR-AUC, ROC-AUC, Recall at K=20.

In [9]:
def recall_at_k(y_true, y_proba, k=0.2):
    n = int(len(y_true) * k)
    top_idx = np.argsort(y_proba)[::-1][:n]
    return y_true[top_idx].sum() / max(y_true.sum(), 1)

mlflow.set_experiment("hospital-readmission-baseline")

with mlflow.start_run(run_name="lr_baseline") as run:
    params = {"max_iter": 500, "random_state": SEED, "class_weight": "balanced"}
    mlflow.log_params(params)
    
    lr = LogisticRegression(**params)
    lr.fit(X_train, y_train)
    y_proba_val = lr.predict_proba(X_val)[:, 1]
    y_pred_val = lr.predict(X_val)
    
    pr_auc = average_precision_score(y_val, y_proba_val)
    roc_auc = roc_auc_score(y_val, y_proba_val)
    rec_k20 = recall_at_k(y_val, y_proba_val, k=0.2)
    
    mlflow.log_metric("pr_auc", pr_auc)
    mlflow.log_metric("roc_auc", roc_auc)
    mlflow.log_metric("recall_at_k20", rec_k20)
    mlflow.set_tag("model", "LogisticRegression")
    
    cm = confusion_matrix(y_val, y_pred_val)
    cm_df = pd.DataFrame(cm, index=['No', 'Yes'], columns=['No', 'Yes'])
    cm_df.to_csv("confusion_matrix_labeled.csv")
    mlflow.log_artifact("confusion_matrix_labeled.csv")
    
    import seaborn as sns
    import matplotlib.pyplot as plt
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues')
    plt.title('Confusion Matrix (LR Baseline)')
    plt.ylabel('True')
    plt.xlabel('Predicted')
    plt.tight_layout()
    plt.savefig('confusion_matrix_heatmap.png')
    plt.close()
    mlflow.log_artifact('confusion_matrix_heatmap.png')
    
    input_example = X_val[0:1]
    if hasattr(input_example, 'toarray'):
        input_example = input_example.toarray()
    mlflow.sklearn.log_model(lr, "model", input_example=input_example)
    
    print(f'Logged: PR-AUC={pr_auc:.3f}, ROC-AUC={roc_auc:.3f}, Recall@K20={rec_k20:.3f}')
    print(f'Run ID: {run.info.run_id}')

/Users/isabelwu/miniconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
2026/03/10 21:37:44 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/10 21:37:44 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn

Logged: PR-AUC=0.189, ROC-AUC=0.644, Recall@K20=0.350
Run ID: bf773b5ca45841a2b07d792505a2151a


## Launch MLflow UI
View experiments at http://localhost:5001

In [ ]:
import subprocess, sys
subprocess.run(["bash", "-c", "lsof -ti :5001 | xargs kill -9 2>/dev/null"], capture_output=True)
subprocess.Popen([sys.executable, '-m', 'mlflow', 'ui', '--port', '5001', '--backend-store-uri', 'sqlite:///mlflow.db'])
print('MLflow UI: http://localhost:5001')

MLflow UI: http://localhost:5001


Registry store URI not provided. Using backend store URI.


[MLflow] Security middleware enabled with default settings (localhost-only). To allow connections from other hosts, use --host 0.0.0.0 and configure --allowed-hosts and --cors-allowed-origins.
INFO:     Uvicorn running on http://127.0.0.1:5001 (Press CTRL+C to quit)
INFO:     Started parent process [44248]
2026/03/10 21:37:54 INFO mlflow.server.jobs.utils: Starting huey consumer for job function optimize_prompts
2026/03/10 21:37:54 INFO mlflow.server.jobs.utils: Starting huey consumer for job function invoke_scorer
2026/03/10 21:37:54 INFO mlflow.server.jobs.utils: Starting huey consumer for job function run_online_trace_scorer
2026/03/10 21:37:54 INFO mlflow.server.jobs.utils: Starting huey consumer for job function run_online_session_scorer
2026/03/10 21:37:54 INFO mlflow.server.jobs.utils: Starting dedicated Huey consumer for periodic tasks
INFO:     Started server process [44254]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Started se

INFO:     127.0.0.1:63813 - "GET / HTTP/1.1" 304 Not Modified
INFO:     127.0.0.1:63813 - "GET /ajax-api/3.0/mlflow/server-info HTTP/1.1" 200 OK
INFO:     127.0.0.1:63813 - "GET /ajax-api/3.0/mlflow/assistant/config HTTP/1.1" 200 OK
INFO:     127.0.0.1:63813 - "GET /ajax-api/2.0/mlflow/experiments/search?max_results=5&order_by=last_update_time+DESC HTTP/1.1" 200 OK


2026/03/10 21:37:58 INFO huey.consumer: Huey consumer started with 10 thread, PID 44291 at 2026-03-10 20:37:58.393972
[2026-03-10 21:37:58,393] INFO:huey.consumer:MainThread:Huey consumer started with 10 thread, PID 44291 at 2026-03-10 20:37:58.393972
2026/03/10 21:37:58 INFO huey.consumer: Huey consumer started with 5 thread, PID 44290 at 2026-03-10 20:37:58.394008
[2026-03-10 21:37:58,394] INFO:huey.consumer:MainThread:Huey consumer started with 5 thread, PID 44290 at 2026-03-10 20:37:58.394008
2026/03/10 21:37:58 INFO huey.consumer: Scheduler runs every 1 second(s).
[2026-03-10 21:37:58,394] INFO:huey.consumer:MainThread:Scheduler runs every 1 second(s).
2026/03/10 21:37:58 INFO huey.consumer: Scheduler runs every 1 second(s).
[2026-03-10 21:37:58,394] INFO:huey.consumer:MainThread:Scheduler runs every 1 second(s).
2026/03/10 21:37:58 INFO huey.consumer: Periodic tasks are enabled.
[2026-03-10 21:37:58,394] INFO:huey.consumer:MainThread:Periodic tasks are enabled.
2026/03/10 21:37:5

...

2026/03/10 21:40:02 INFO huey: Executing mlflow.server.jobs.utils.online_scoring_scheduler: 17d79613-805b-4a53-813d-eeecc6052d25
[2026-03-10 21:40:02,875] INFO:huey:Worker-3:Executing mlflow.server.jobs.utils.online_scoring_scheduler: 17d79613-805b-4a53-813d-eeecc6052d25
2026/03/10 21:40:02 INFO huey: mlflow.server.jobs.utils.online_scoring_scheduler: 17d79613-805b-4a53-813d-eeecc6052d25 executed in 0.032s
[2026-03-10 21:40:02,908] INFO:huey:Worker-3:mlflow.server.jobs.utils.online_scoring_scheduler: 17d79613-805b-4a53-813d-eeecc6052d25 executed in 0.032s
2026/03/10 21:40:58 INFO huey.consumer.Scheduler: Enqueueing periodic task mlflow.server.jobs.utils.online_scoring_scheduler: 5d2ee19b-87b0-4d4b-9ca4-557139201653.
[2026-03-10 21:40:58,422] INFO:huey.consumer.Scheduler:Scheduler:Enqueueing periodic task mlflow.server.jobs.utils.online_scoring_scheduler: 5d2ee19b-87b0-4d4b-9ca4-557139201653.
2026/03/10 21:41:00 INFO huey: Executing mlflow.server.jobs.utils.online_scoring_scheduler: 5d2

INFO:     127.0.0.1:64115 - "POST /ajax-api/2.0/mlflow/runs/search HTTP/1.1" 200 OK


2026/03/10 21:41:58 INFO huey.consumer.Scheduler: Enqueueing periodic task mlflow.server.jobs.utils.online_scoring_scheduler: 40ab5319-15d1-4898-9e7a-43f514e58a97.
[2026-03-10 21:41:58,417] INFO:huey.consumer.Scheduler:Scheduler:Enqueueing periodic task mlflow.server.jobs.utils.online_scoring_scheduler: 40ab5319-15d1-4898-9e7a-43f514e58a97.
2026/03/10 21:41:58 INFO huey: Executing mlflow.server.jobs.utils.online_scoring_scheduler: 40ab5319-15d1-4898-9e7a-43f514e58a97
[2026-03-10 21:41:58,638] INFO:huey:Worker-3:Executing mlflow.server.jobs.utils.online_scoring_scheduler: 40ab5319-15d1-4898-9e7a-43f514e58a97
2026/03/10 21:41:58 INFO huey: mlflow.server.jobs.utils.online_scoring_scheduler: 40ab5319-15d1-4898-9e7a-43f514e58a97 executed in 0.017s
[2026-03-10 21:41:58,655] INFO:huey:Worker-3:mlflow.server.jobs.utils.online_scoring_scheduler: 40ab5319-15d1-4898-9e7a-43f514e58a97 executed in 0.017s
2026/03/10 21:42:58 INFO huey.consumer.Scheduler: Enqueueing periodic task mlflow.server.jobs.u

INFO:     127.0.0.1:64341 - "GET /ajax-api/2.0/mlflow/experiments/search?max_results=25&order_by=last_update_time+DESC&filter=tags.%60mlflow.experiment.isGateway%60+IS+NULL HTTP/1.1" 200 OK


2026/03/10 21:46:58 INFO huey.consumer.Scheduler: Enqueueing periodic task mlflow.server.jobs.utils.online_scoring_scheduler: eb708f85-eccf-40fe-a9d8-34507be9f204.
[2026-03-10 21:46:58,418] INFO:huey.consumer.Scheduler:Scheduler:Enqueueing periodic task mlflow.server.jobs.utils.online_scoring_scheduler: eb708f85-eccf-40fe-a9d8-34507be9f204.
2026/03/10 21:47:05 INFO huey: Executing mlflow.server.jobs.utils.online_scoring_scheduler: eb708f85-eccf-40fe-a9d8-34507be9f204
[2026-03-10 21:47:05,257] INFO:huey:Worker-1:Executing mlflow.server.jobs.utils.online_scoring_scheduler: eb708f85-eccf-40fe-a9d8-34507be9f204
2026/03/10 21:47:05 INFO huey: mlflow.server.jobs.utils.online_scoring_scheduler: eb708f85-eccf-40fe-a9d8-34507be9f204 executed in 0.013s
[2026-03-10 21:47:05,270] INFO:huey:Worker-1:mlflow.server.jobs.utils.online_scoring_scheduler: eb708f85-eccf-40fe-a9d8-34507be9f204 executed in 0.013s
2026/03/10 21:47:58 INFO huey.consumer.Scheduler: Enqueueing periodic task mlflow.server.jobs.u